# 16 — Database Inspection

Quick look at every table in `hotel_reviews.db`: schema, row counts, and sample rows.

In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

DB = "../data/hotel_reviews.db"
con = duckdb.connect(DB, read_only=True)
print("Connected:", DB)

## Section 1 — Table Overview

In [ ]:
tables = [r[0] for r in con.execute("SHOW TABLES").fetchall()]

rows = []
for t in tables:
    count = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    cols  = [r[0] for r in con.execute(f"DESCRIBE {t}").fetchall()]
    rows.append({"table": t, "rows": count, "columns": len(cols), "column_names": ", ".join(cols)})

overview = pd.DataFrame(rows)
overview

## Section 2 — Schema per Table

In [ ]:
for t in tables:
    print(f"\n{'─'*60}  {t}")
    schema = con.execute(f"DESCRIBE {t}").df()
    display(schema[["column_name", "column_type", "null"]])

## Section 3 — Sample Rows per Table

5 rows from each table. `REVIEW_EMBEDDINGS` shows only the first 4 embedding values to keep output readable.

In [ ]:
for t in tables:
    print(f"\n{'─'*60}  {t}")
    if t == "REVIEW_EMBEDDINGS":
        df = con.execute(
            "SELECT review_id, embedding[1:4] AS embedding_preview FROM REVIEW_EMBEDDINGS LIMIT 5"
        ).df()
    else:
        df = con.execute(f"SELECT * FROM {t} LIMIT 5").df()
    display(df)

## Section 4 — REVIEW_TOPICS Deep Dive

Check which `run_id` values exist and how many reviews and topics each run has.

In [ ]:
rt_summary = con.execute("""
    SELECT
        run_id,
        COUNT(*)                                        AS total_assignments,
        COUNT(DISTINCT review_id)                       AS unique_reviews,
        COUNT(DISTINCT topic_id)                        AS unique_topics,
        SUM(CASE WHEN topic_id = -1 THEN 1 ELSE 0 END) AS outlier_count,
        ROUND(AVG(prob), 3)                             AS avg_prob
    FROM REVIEW_TOPICS
    GROUP BY run_id
    ORDER BY run_id
""").df()

rt_summary

## Section 5 — TOPIC_LABELS Deep Dive

Check the current label schema, which runs are covered, and what `seed_topic` values look like.

In [ ]:
# Topics per run
tl_runs = con.execute("""
    SELECT
        run_id,
        COUNT(*)                                           AS total_topics,
        SUM(CASE WHEN topic_id = -1 THEN 1 ELSE 0 END)   AS has_outlier_row,
        COUNT(DISTINCT seed_topic)                         AS distinct_seed_labels,
        STRING_AGG(DISTINCT seed_topic, ', ')              AS seed_labels
    FROM TOPIC_LABELS
    GROUP BY run_id
    ORDER BY run_id
""").df()

tl_runs

In [ ]:
# Seed label distribution across all runs
tl_dist = con.execute("""
    SELECT
        seed_topic,
        COUNT(*)              AS topic_count,
        ROUND(AVG(seed_score), 3) AS avg_confidence,
        ROUND(MIN(seed_score), 3) AS min_confidence,
        ROUND(MAX(seed_score), 3) AS max_confidence
    FROM TOPIC_LABELS
    WHERE topic_id != -1
    GROUP BY seed_topic
    ORDER BY topic_count DESC
""").df()

tl_dist

In [ ]:
# Sample 10 rows from TOPIC_LABELS to inspect top_words and seed info
con.execute("""
    SELECT run_id, topic_id, n_docs, seed_topic, seed_score, top_words
    FROM TOPIC_LABELS
    WHERE topic_id != -1
    ORDER BY run_id, n_docs DESC
    LIMIT 10
""").df()

## Section 6 — REVIEW_DATA Quick Stats

In [ ]:
# Language split
print("── Language split ──")
display(con.execute("""
    SELECT language, COUNT(*) AS reviews, ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 1) AS pct
    FROM REVIEW_DATA GROUP BY language ORDER BY reviews DESC
""").df())

# Source split
print("\n── Source split ──")
display(con.execute("""
    SELECT source, COUNT(*) AS reviews FROM REVIEW_DATA GROUP BY source
""").df())

# Year distribution
print("\n── Year distribution ──")
display(con.execute("""
    SELECT review_year, COUNT(*) AS reviews
    FROM REVIEW_DATA WHERE review_year IS NOT NULL
    GROUP BY review_year ORDER BY review_year
""").df().T)

In [ ]:
# Distance band distribution
print("── Distance band distribution ──")
display(con.execute("""
    SELECT
        CASE
            WHEN distance2coastline < 0.1  THEN 'A — Beachfront (< 0.1 km)'
            WHEN distance2coastline < 1.0  THEN 'B — Near-coast (0.1–1.0 km)'
            WHEN distance2coastline >= 1.0 THEN 'C — Inland (≥ 1.0 km)'
            ELSE 'NULL'
        END AS band,
        COUNT(*) AS reviews,
        ROUND(COUNT(*)*100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM REVIEW_DATA
    GROUP BY band
    ORDER BY band
""").df())

## Section 7 — Silver-Label Readiness Check

Confirms whether `TOPIC_LABELS` already has the new silver-label columns (`key_aspect`, `optional_aspects`, etc.) or still uses the old `seed_topic` / `seed_score` schema.

In [ ]:
SILVER_COLS = {"key_aspect", "optional_aspects", "sentiment", "confidence", "evidence_keywords", "short_reason"}
OLD_COLS    = {"seed_topic", "seed_score"}

existing = {r[0] for r in con.execute("DESCRIBE TOPIC_LABELS").fetchall()}

print("TOPIC_LABELS columns:", sorted(existing))
print()

has_silver = SILVER_COLS.issubset(existing)
has_old    = OLD_COLS.issubset(existing)

if has_silver:
    print("✅  Silver-label columns present — ready for analysis.")
    missing = SILVER_COLS - existing
    if missing:
        print(f"   Missing: {missing}")
else:
    print("⚠️   Silver-label columns NOT present.")
    print(f"   Missing: {SILVER_COLS - existing}")

if has_old:
    print("ℹ️   Old seed columns (seed_topic, seed_score) still present.")

# Check for any NULL key_aspect if the column exists
if "key_aspect" in existing:
    nulls = con.execute("SELECT COUNT(*) FROM TOPIC_LABELS WHERE key_aspect IS NULL AND topic_id != -1").fetchone()[0]
    total = con.execute("SELECT COUNT(*) FROM TOPIC_LABELS WHERE topic_id != -1").fetchone()[0]
    print(f"\n   key_aspect filled: {total - nulls}/{total} non-outlier topics")

In [ ]:
con.close()
print("Connection closed.")